# 05 - Statistical Analysis, Hypothesis Testing & Thesis Publication Figures

This notebook brings together the **Statistical Hypothesis Testing Engine** and the **Publication Figure & Storyboard Generator** into a unified scientific analysis pipeline.

### 🔬 Part I: Statistical Hypothesis Testing & Reporting
1. **Omnibus Kruskal-Wallis Test**: Multi-group non-parametric difference test across all solvers per problem condition.
2. **Pairwise Mann-Whitney U Tests with FDR**: Two-sided comparisons with **Benjamini-Hochberg False Discovery Rate** correction ($\alpha = 0.05$).
3. **Effect Size Estimation**: Vargha-Delaney $\hat{A}_{12}$ stochastic dominance metric.
4. **Synthesis Transfer Correlation**: Pearson $r, p$ connecting synthesis fitness to empirical benchmark error.
5. **Master Markdown Report**: Automated scientific summary exported to `results/reports/comprehensive_master_report.md`.

---
### 📊 Part II: Thesis Publication Figures & Visual Storyboard
- **Figure E**: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension).
- **Figure 1 (RQ1)**: Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2/RQ3)**: Empirical Convergence Trajectories & Target Precision ECDFs with IQR shaded bounds.
- **Figure 3 (RQ3 Hero)**: Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation)**: Prompt Scaffolding Ablation across LLM model families.

In [13]:
# Ensure project root src/ is in sys.path
from collections import defaultdict
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shared.config import RESULTS_DIR
from shared.database import create_db_session_factory
from benchmarking.infra.storage import SQLiteBenchmarkReadRepository
from benchmarking.infra.io.trace_repository import IOHTraceReader
from benchmarking.application.statistical_service import StatisticalEvaluationService
from benchmarking.domain.services.taxonomy import (
    BBOB_CLASSES,
    BBOB_NAMES,
    get_bbob_class,
    get_bbob_name,
)
from benchmarking.domain import BenchmarkCondition, BenchmarkDataset, RunTrace
from benchmarking.domain.services.resolvers import (
    resolve_canonical_model_slug,
    resolve_folder_solver_name,
)

session_factory = create_db_session_factory()
sqlite_repo = SQLiteBenchmarkReadRepository(session_factory)
trace_reader = IOHTraceReader()
service = StatisticalEvaluationService(sqlite_repo=sqlite_repo, trace_repo=trace_reader)

# ── 1. Thematic Publication Subdirectories ──────────────────────────────────
PUBLICATION_DIR   = RESULTS_DIR / "publication"
MAIN_RESULTS_DIR  = PUBLICATION_DIR / "main_results"
ABLATION_DIR      = PUBLICATION_DIR / "ablation"
EFFECT_SIZES_DIR  = PUBLICATION_DIR / "effect_sizes"
NOISE_DIR         = PUBLICATION_DIR / "noise_robustness"
CONVERGENCE_DIR   = PUBLICATION_DIR / "convergence"
PROFILES_DIR      = RESULTS_DIR / "figures" / "profiles"
STATISTICS_DIR    = RESULTS_DIR / "statistics"

for d in [MAIN_RESULTS_DIR, ABLATION_DIR, EFFECT_SIZES_DIR, NOISE_DIR, CONVERGENCE_DIR, PROFILES_DIR, STATISTICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 2. Modern Typography & Curated Color Palettes ──────────────────────────
FONT_FAMILY = "Inter, -apple-system, BlinkMacSystemFont, Arial, sans-serif"

MODERN_SOLVER_PALETTE = {
    "CMA-ES": "#1E293B",
    "DE": "#475569",
    "PSO": "#94A3B8",
    "LLaMEA-14B / guided": "#0284C7",
    "LLaMEA-14B / thinking": "#059669",
    "LLaMEA-14B / baseline": "#D97706",
    "LLaMEA-14B / vectorization": "#DC2626",
    "LLaMEA-7B / guided": "#7DD3FC",
    "LLaMEA-7B / thinking": "#6EE7B7",
    "LLaMEA-7B / baseline": "#FCD34D",
    "LLaMEA-7B / vectorization": "#FCA5A5",
}

def get_solver_color(s: str) -> str:
    return MODERN_SOLVER_PALETTE.get(s, "#64748B")

print("✅ Statistical service and thematic publication directories initialized.")
REPORTS_DIR       = RESULTS_DIR / "reports"
EVALUATIONS_DIR   = RESULTS_DIR / "ioh_traces"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None

def build_dynamic_solver_palette(solvers):
    return {s: get_solver_color(s) for s in solvers}



✅ Statistical service and thematic publication directories initialized.


## Part I: Statistical Hypothesis Testing & Reporting
Ingest empirical benchmark traces and conduct rigorous non-parametric hypothesis testing with FDR control.

In [14]:
# Ingest benchmark traces and synthesis records
df_exp, df_iter = service.get_synthesis_dataframes()
all_benchmark_data = service.load_evaluation_traces(
    dims=FILTER_DIMS,
    problems=FILTER_PROBLEMS,
    noise_stds=FILTER_NOISE_STDS,
    solver_resolver=resolve_folder_solver_name,
)

if not all_benchmark_data:
    raise RuntimeError(f'No benchmark traces found in {EVALUATIONS_DIR}!')

all_dims = all_benchmark_data.dims
all_noise_stds = all_benchmark_data.noise_stds
clean_std = 0.0 if 0.0 in all_noise_stds else (all_noise_stds[0] if all_noise_stds else 0.0)
noisy_std = next((n for n in all_noise_stds if n > 0.0), all_noise_stds[-1] if all_noise_stds else 0.05)
PROBLEM_IDS = all_benchmark_data.problem_ids

DISCOVERED_SOLVERS = all_benchmark_data.solvers
SOLVER_PALETTE = build_dynamic_solver_palette(DISCOVERED_SOLVERS)

MODELS_TO_SOLVERS = defaultdict(list)
for s in DISCOVERED_SOLVERS:
    if ' / ' in s:
        MODELS_TO_SOLVERS[s.split(' / ')[0]].append(s)

LLM_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' in s]
CLASSICAL_SOLVERS_ORDER = [s for s in DISCOVERED_SOLVERS if ' / ' not in s]
ALL_SOLVERS_ORDER = LLM_SOLVERS_ORDER + CLASSICAL_SOLVERS_ORDER

print(f'📦 Loaded {len(df_exp)} experiments and {len(all_benchmark_data)} problem conditions.')
print(f'🎯 Problems: {PROBLEM_IDS} | Dimensions: {all_dims} | Solvers: {DISCOVERED_SOLVERS}')


2026-08-26 22:27:33 INFO TemporaryDirectory.cleanup() worked.
2026-08-26 22:27:33 INFO shutil.rmtree worked.


📦 Loaded 292 experiments and 30 problem conditions.
🎯 Problems: [1, 8, 11, 15, 21] | Dimensions: [2, 3, 5] | Solvers: ['CMA-ES', 'DE', 'LLaMEA-14B / baseline', 'LLaMEA-14B / guided', 'LLaMEA-14B / thinking', 'LLaMEA-14B / vectorization', 'LLaMEA-7B / baseline', 'LLaMEA-7B / guided', 'LLaMEA-7B / thinking', 'LLaMEA-7B / vectorization', 'PSO']


In [15]:
# ── 1. Omnibus Kruskal-Wallis & Pairwise FDR Tests ─────────────────────────
df_omnibus = service.run_omnibus_kruskal(all_benchmark_data)
df_pairwise = service.run_pairwise_fdr(all_benchmark_data, alpha=0.05)
r_val, p_val = service.compute_synthesis_transfer_correlation(df_exp)

print(f'✅ Omnibus Tests: {len(df_omnibus)} rows ({len(df_omnibus[df_omnibus["Significant"] == "Yes"])} significant)')
print(f'✅ Pairwise FDR Tests: {len(df_pairwise)} rows ({len(df_pairwise[df_pairwise["Significant (FDR)"]])} significant)')
print(f'✅ Synthesis Transfer Correlation: r = {r_val:.3f} (p = {p_val:.3e})')

# Export Master Markdown Report
report_path = REPORTS_DIR / 'comprehensive_master_report.md'
service.generate_markdown_report(df_omnibus=df_omnibus, df_pairwise=df_pairwise, df_exp=df_exp, output_path=report_path)
print(f'🎉 Master Comprehensive Report generated: {report_path}')


✅ Omnibus Tests: 30 rows (30 significant)
✅ Pairwise FDR Tests: 1630 rows (1241 significant)
✅ Synthesis Transfer Correlation: r = 0.000 (p = 1.000e+00)
🎉 Master Comprehensive Report generated: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/reports/comprehensive_master_report.md


## Part II: Thesis Publication Figures & Visual Storyboard
Render high-DPI thesis figures and storyboard artifacts.

# ── Figure E: Problem Difficulty & Noise Sensitivity Dashboard (Per Dimension) ──
clean_std = 0.0
noisy_std = 0.05

for dim in all_dims:


In [16]:
# ── Figure 5: Landscape Fragility Matrix (Clean → Noisy Degradation) ─────────
for dim in all_dims:
    frag_matrix, p_labels = service.compute_fragility_matrix(
        all_benchmark_data, dim, ALL_SOLVERS_ORDER, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig5 = go.Figure(data=go.Heatmap(
        z=frag_matrix,
        x=ALL_SOLVERS_ORDER,
        y=p_labels,
        colorscale="RdBu",
        zmid=0,
        colorbar=dict(title="<b>Fragility Δ</b>", thickness=12, len=0.85)
    ))
    fig5.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 5: Landscape Fragility Matrix (Clean → Noisy Degradation) — {dim}D</b><br><sup>Performance Drop (Δ = Clean Success Rate - Noisy Success Rate) Across Solvers by BBOB Problem</sup>",
            font=dict(size=14, color="#1E293B", family=FONT_FAMILY),
            x=0.02, y=0.96
        ),
        width=1050, height=480,
        margin=dict(l=160, r=40, t=95, b=100),
        xaxis=dict(tickangle=-30, tickfont=dict(size=10, family=FONT_FAMILY)),
        yaxis=dict(tickfont=dict(size=11, family=FONT_FAMILY))
    )
    out_p = NOISE_DIR / f"fig_05_noise_fragility_matrix_{dim}D.png"
    fig5.write_image(str(out_p), scale=3)

print("✅ Figure 5 (Fragility Matrix) generated in results/publication/noise_robustness/")


2026-08-26 22:27:34 INFO Chromium init'ed with kwargs {}
2026-08-26 22:27:34 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:27:34 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmphet6_kf_.
2026-08-26 22:27:34 INFO Opening browser.
2026-08-26 22:27:34 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpus07q3r4.
2026-08-26 22:27:34 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpus07q3r4
2026-08-26 22:27:37 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmphet6_kf_/index.html
2026-08-26 22:27:38 INFO Getting tab from queue (has 1)
2026-08-26 22:27:38 INFO Got E11B
2026-08-26 22:27:38 INFO Reloading tab E11B before return.
2026-08-26 22:27:38 INFO Putting tab E11B back (queue size: 0).
2026-08-26 22:27:38 INFO Waiting for all cleanups to finish.
2026-08-26 22:27:38 INFO Exiting Kaleido.
2026-08-26 22:27:38 INFO T

✅ Figure 5 (Fragility Matrix) generated in results/publication/noise_robustness/


### 📊 Model-Specific Hardness Success Rates (Clean vs. Noisy)
Separates the mean success rate analysis per LLM model (, ) across clean and noisy landscapes.

In [17]:
# ── Model-Specific Success Rate by Landscape Hardness (Clean vs. Noisy) ──
def render_model_success_rate_by_hardness(model_tag: str, solvers_list: list, dim: int):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"<b>(A) Deterministic Landscape (σ={clean_std}, {dim}D)</b>",
            f"<b>(B) Noisy Stochastic Landscape (σ={noisy_std}, {dim}D)</b>"
        ),
        horizontal_spacing=0.10
    )
    
    for c_idx, noise_level in enumerate([clean_std, noisy_std], start=1):
        df_hard = service.compute_hardness_success_rates(all_benchmark_data, dim, solvers_list, noise_level=noise_level)
        for solver in solvers_list:
            sub_s = df_hard[df_hard["Solver"] == solver]
            if not sub_s.empty:
                is_classical = " / " not in solver
                fig.add_trace(
                    go.Bar(
                        name=solver,
                        x=sub_s["Class"],
                        y=sub_s["Success Rate"],
                        marker=dict(
                            color=get_solver_color(solver),
                            line=dict(color="#0F172A", width=0.8)
                        ),
                        showlegend=(c_idx == 1)
                    ),
                    row=1, col=c_idx
                )
                
    fig.update_xaxes(tickangle=-15, tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=1)
    fig.update_xaxes(tickangle=-15, tickfont=dict(size=10, family=FONT_FAMILY), row=1, col=2)
    fig.update_yaxes(title="<b>Target Success Rate (Δy ≤ 10⁻⁸)</b>", range=[0, 1.08], showgrid=True, gridcolor="#F1F5F9", row=1, col=1)
    fig.update_yaxes(range=[0, 1.08], showgrid=True, gridcolor="#F1F5F9", row=1, col=2)
    
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Empirical Success Rate by BBOB Landscape Hardness — {model_tag.upper()} ({dim}D)</b><br><sup>Comparison of Target Precision Hitting Rates Across 5 Problem Classes in Deterministic vs. Noisy Regimes</sup>",
            x=0.02, y=0.96,
            font=dict(size=14, color="#1E293B", family=FONT_FAMILY)
        ),
        barmode="group",
        bargap=0.25,
        bargroupgap=0.08,
        width=1220, height=560,
        margin=dict(l=70, r=40, t=100, b=120),
        legend=dict(
            orientation="h",
            yanchor="top", y=-0.22,
            xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)",
            bordercolor="#E2E8F0",
            borderwidth=1,
            font=dict(size=11, family=FONT_FAMILY)
        )
    )
    
    slug = resolve_canonical_model_slug(model_tag)
    m_dir = PROFILES_DIR / slug / f"{dim}D"
    m_dir.mkdir(parents=True, exist_ok=True)
    out_p = m_dir / "figure_success_rate_by_hardness.png"
    fig.write_image(str(out_p), scale=3)

for dim in all_dims:
    for model_name, solvers_list in MODELS_TO_SOLVERS.items():
        solvers_to_plot = solvers_list + CLASSICAL_SOLVERS_ORDER
        render_model_success_rate_by_hardness(model_name, solvers_to_plot, dim)

print("✅ Model-specific success rate by hardness generated for all models and dimensions.")


2026-08-26 22:27:42 INFO Chromium init'ed with kwargs {}
2026-08-26 22:27:42 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:27:42 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp3vk5ks00.
2026-08-26 22:27:42 INFO Opening browser.
2026-08-26 22:27:42 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpucrjl0sk.
2026-08-26 22:27:42 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpucrjl0sk
2026-08-26 22:27:43 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp3vk5ks00/index.html
2026-08-26 22:27:45 INFO Getting tab from queue (has 1)
2026-08-26 22:27:45 INFO Got 257E
2026-08-26 22:27:45 INFO Reloading tab 257E before return.
2026-08-26 22:27:45 INFO Putting tab 257E back (queue size: 0).
2026-08-26 22:27:45 INFO Waiting for all cleanups to finish.
2026-08-26 22:27:45 INFO Exiting Kaleido.
2026-08-26 22:27:45 INFO T

✅ Model-specific success rate by hardness generated for all models and dimensions.


# 🎓 Part II: Thesis Visual Storyboard (RQ1 → RQ2 → RQ3 → Scaffolding Narrative Chain)

The following four figures form the core visual evidence for the thesis, saved into `results/figures/{dim}D/thesis/`:
- **Figure 1 (RQ1):** Benchmark Stochastic Extension Validation (Clean vs. Noisy degradation per problem).
- **Figure 2 (RQ2):** LLaMEA Synthesis Competency vs. Classical Baselines (Clean Convergence Trajectories & IQR).
- **Figure 3 (RQ3 Hero):** Cross-Environment Noise Robustness Profile (Clean vs. Noisy success rate drops).
- **Figure 4 (RQ2/3 Ablation):** Prompt Scaffolding Ablation on LLaMEA-14B (Baseline vs. Guided vs. Thinking vs. Vectorization).


In [18]:
# ── THESIS Figure 1: Benchmark Difficulty under Noise Extension ──────────────
for dim in all_dims:
    clean_meds, noisy_meds, p_labels = service.compute_validation_medians(
        all_benchmark_data, dim, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig1 = go.Figure()
    fig1.add_trace(go.Bar(
        name=f"Deterministic (σ={clean_std})",
        x=p_labels,
        y=np.maximum(clean_meds, 1e-16),
        marker=dict(color="#1E3A8A", line=dict(color="#0F172A", width=1.2))
    ))
    fig1.add_trace(go.Bar(
        name=f"Noisy Stochastic (σ={noisy_std})",
        x=p_labels,
        y=np.maximum(noisy_meds, 1e-16),
        marker=dict(
            color="#EA580C",
            pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.4, size=8),
            line=dict(color="#7C2D12", width=1.2)
        )
    ))
    fig1.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 1: Benchmark Problem Difficulty under Stochastic Noise — {dim}D</b><br><sup>Median Terminal Optimization Error (Δy) Across All Solvers by BBOB Landscape Class</sup>",
            font=dict(size=14, color="#1E293B", family=FONT_FAMILY),
            x=0.02, y=0.96
        ),
        xaxis=dict(title="<b>BBOB Landscape Class</b>", tickfont=dict(size=11, family=FONT_FAMILY)),
        yaxis=dict(
            type="log", title="<b>Median Final Error log₁₀(Δy)</b>", range=[-16, 4],
            showgrid=True, gridwidth=1, gridcolor="#F1F5F9"
        ),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1050, height=540,
        margin=dict(l=75, r=40, t=100, b=90),
        legend=dict(
            orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
            font=dict(size=11, family=FONT_FAMILY)
        )
    )
    out_p = MAIN_RESULTS_DIR / f"fig_01_benchmark_difficulty_{dim}D.png"
    fig1.write_image(str(out_p), scale=3)

print("✅ Figure 1 (Benchmark Difficulty) generated in results/publication/main_results/")


2026-08-26 22:27:56 INFO Chromium init'ed with kwargs {}
2026-08-26 22:27:56 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:27:56 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppl7r1c_9.
2026-08-26 22:27:56 INFO Opening browser.
2026-08-26 22:27:56 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpbf5wnlb2.
2026-08-26 22:27:56 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpbf5wnlb2
2026-08-26 22:27:57 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmppl7r1c_9/index.html
2026-08-26 22:27:57 INFO TemporaryDirectory.cleanup() worked.
2026-08-26 22:27:57 INFO shutil.rmtree worked.
2026-08-26 22:27:57 INFO TemporaryDirectory.cleanup() worked.
2026-08-26 22:27:57 INFO shutil.rmtree worked.
2026-08-26 22:27:57 INFO TemporaryDirectory.cleanup() worked.
2026-08-26 22:27:57 INFO shutil.rmtree worked.
2026-08-26 22:2

✅ Figure 1 (Benchmark Difficulty) generated in results/publication/main_results/


In [19]:
# ── THESIS: Multi-Panel Model Convergence & Target Precision ECDFs ──────────
targets = np.logspace(-8, 2, 100)
eval_grid = np.logspace(0, 5, 200)

for dim in all_dims:
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        slug = resolve_canonical_model_slug(model_name)
        m_dir = PROFILES_DIR / slug / f"{dim}D"
        m_dir.mkdir(parents=True, exist_ok=True)
        solvers_to_plot = s_list + CLASSICAL_SOLVERS_ORDER
        for n_std, label_env in [(clean_std, "Deterministic (σ=0.0)"), (noisy_std, "Noisy Stochastic (σ=0.05)")]:
            env_dir = m_dir / f"std_{n_std}"
            env_dir.mkdir(parents=True, exist_ok=True)
            coords = [((i // 3) + 1, (i % 3) + 1) for i in range(len(PROBLEM_IDS) + 1)]
            subplot_titles = [f"<b>{BBOB_NAMES.get(p, f'f{p}')} ({BBOB_CLASSES.get(p, '')})</b>" for p in PROBLEM_IDS] + ["<b>Overall Aggregate Summary</b>"]
            
            # 1. Multi-panel convergence trajectories (Log-scale Median + IQR)
            fig_m = make_subplots(
                rows=2, cols=3,
                subplot_titles=subplot_titles,
                vertical_spacing=0.18,
                horizontal_spacing=0.08
            )
            for idx, p_id in enumerate(PROBLEM_IDS):
                r_idx, c_idx = coords[idx]
                cond = BenchmarkCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                s_dict = all_benchmark_data.get(cond, {})
                for s_name in solvers_to_plot:
                    runs = s_dict.get(s_name, [])
                    med, q25, q75, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                    if not np.isnan(med).all():
                        color = get_solver_color(s_name)
                        is_classical = " / " not in s_name
                        fig_m.add_trace(go.Scatter(
                            x=eval_grid, y=med, mode="lines", name=s_name,
                            line=dict(color=color, width=2.4 if "14B" in s_name or is_classical else 1.8, dash="dash" if is_classical else "solid"),
                            showlegend=(idx == 0)
                        ), row=r_idx, col=c_idx)
                        fig_m.add_trace(go.Scatter(
                            x=np.concatenate([eval_grid, eval_grid[::-1]]),
                            y=np.concatenate([q75, q25[::-1]]),
                            fill="toself",
                            fillcolor=color.replace("rgb", "rgba").replace(")", ", 0.12)") if "rgb" in color else "rgba(100,116,139,0.12)",
                            line=dict(color="rgba(255,255,255,0)"),
                            showlegend=False, hoverinfo="skip"
                        ), row=r_idx, col=c_idx)
                
                fig_m.update_xaxes(type="log", title_text="<b>Evaluations</b>", title_font=dict(size=10), showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
                fig_m.update_yaxes(type="log", title_text="<b>Error Δy</b>", title_font=dict(size=10), range=[-16, 4], showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
            
            # Panel 6 for fig_m: Overall Convergence (Median across problems)
            r_idx6, c_idx6 = coords[len(PROBLEM_IDS)]
            for s_name in solvers_to_plot:
                all_p_meds = []
                for p_id in PROBLEM_IDS:
                    cond = BenchmarkCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                    runs = all_benchmark_data.get(cond, {}).get(s_name, [])
                    if runs:
                        med, _, _, _ = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        all_p_meds.append(med)
                if all_p_meds:
                    mean_med = np.nanmean(all_p_meds, axis=0)
                    color = get_solver_color(s_name)
                    is_classical = " / " not in s_name
                    fig_m.add_trace(go.Scatter(
                        x=eval_grid, y=mean_med, mode="lines", name=s_name,
                        line=dict(color=color, width=2.4 if "14B" in s_name or is_classical else 1.8, dash="dash" if is_classical else "solid"),
                        showlegend=False
                    ), row=r_idx6, col=c_idx6)
            fig_m.update_xaxes(type="log", title_text="<b>Evaluations</b>", title_font=dict(size=10), showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            fig_m.update_yaxes(type="log", title_text="<b>Mean Error Δy</b>", title_font=dict(size=10), range=[-16, 4], showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            
            fig_m.update_layout(
                template="plotly_white",
                title=dict(
                    text=f"<b>Empirical Convergence Trajectories [{label_env}] — {slug.upper()} ({dim}D)</b><br><sup>Log-scale Median Convergence with Shaded IQR (25th–75th Percentiles) Across 5 BBOB Problem Classes vs. Classical Baselines</sup>",
                    font=dict(size=14, color="#1E293B", family=FONT_FAMILY),
                    x=0.02, y=0.97
                ),
                width=1320, height=840,
                margin=dict(l=70, r=40, t=110, b=120),
                legend=dict(
                    orientation="h",
                    yanchor="top", y=-0.14,
                    xanchor="center", x=0.5,
                    bgcolor="rgba(255,255,255,0.95)",
                    bordercolor="#E2E8F0",
                    borderwidth=1,
                    font=dict(size=11, family=FONT_FAMILY)
                )
            )
            out_m_traj = env_dir / "convergence_trajectories.png"
            fig_m.write_image(str(out_m_traj), scale=3)

            # 2. Multi-panel Target Precision ECDF (5 Problems + Overall Aggregate)
            fig_ecdf = make_subplots(
                rows=2, cols=3,
                subplot_titles=subplot_titles,
                vertical_spacing=0.18,
                horizontal_spacing=0.08
            )
            for idx, p_id in enumerate(PROBLEM_IDS):
                r_idx, c_idx = coords[idx]
                cond = BenchmarkCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                s_dict = all_benchmark_data.get(cond, {})
                for s_name in solvers_to_plot:
                    runs = s_dict.get(s_name, [])
                    if runs:
                        _, _, _, ecdf_curve = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        color = get_solver_color(s_name)
                        is_classical = " / " not in s_name
                        fig_ecdf.add_trace(go.Scatter(
                            x=targets, y=ecdf_curve, mode="lines", name=s_name,
                            line=dict(color=color, width=2.4 if "14B" in s_name or is_classical else 1.8, dash="dash" if is_classical else "solid"),
                            showlegend=(idx == 0)
                        ), row=r_idx, col=c_idx)
                fig_ecdf.update_xaxes(type="log", title_text="<b>Target Precision Δy</b>", title_font=dict(size=10), autorange="reversed", showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
                fig_ecdf.update_yaxes(title_text="<b>Proportion Solved</b>", title_font=dict(size=10), range=[-0.02, 1.05], showgrid=True, gridcolor="#F1F5F9", row=r_idx, col=c_idx)
            
            # Panel 6: Overall Aggregate ECDF
            r_idx6, c_idx6 = coords[len(PROBLEM_IDS)]
            for s_name in solvers_to_plot:
                all_problem_ecdfs = []
                for p_id in PROBLEM_IDS:
                    cond = BenchmarkCondition(dim=dim, noise_std=n_std, problem_id=p_id)
                    runs = all_benchmark_data.get(cond, {}).get(s_name, [])
                    if runs:
                        _, _, _, ecdf_curve = service.compute_trajectory_and_ecdf(runs, eval_grid, targets)
                        all_problem_ecdfs.append(ecdf_curve)
                if all_problem_ecdfs:
                    mean_ecdf = np.mean(all_problem_ecdfs, axis=0)
                    color = get_solver_color(s_name)
                    is_classical = " / " not in s_name
                    fig_ecdf.add_trace(go.Scatter(
                        x=targets, y=mean_ecdf, mode="lines", name=s_name,
                        line=dict(color=color, width=2.4 if "14B" in s_name or is_classical else 1.8, dash="dash" if is_classical else "solid"),
                        showlegend=False
                    ), row=r_idx6, col=c_idx6)
            fig_ecdf.update_xaxes(type="log", title_text="<b>Target Precision Δy</b>", title_font=dict(size=10), autorange="reversed", showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            fig_ecdf.update_yaxes(title_text="<b>Overall Proportion</b>", title_font=dict(size=10), range=[-0.02, 1.05], showgrid=True, gridcolor="#F1F5F9", row=r_idx6, col=c_idx6)
            
            fig_ecdf.update_layout(
                template="plotly_white",
                title=dict(
                    text=f"<b>Empirical Cumulative Distribution Functions [{label_env}] — {slug.upper()} ({dim}D)</b><br><sup>Fraction of Solved Targets (10⁻⁸ ≤ Δy ≤ 10²) Across 5 BBOB Problem Classes vs. Classical Baselines</sup>",
                    font=dict(size=14, color="#1E293B", family=FONT_FAMILY),
                    x=0.02, y=0.97
                ),
                width=1320, height=840,
                margin=dict(l=70, r=40, t=110, b=120),
                legend=dict(
                    orientation="h",
                    yanchor="top", y=-0.14,
                    xanchor="center", x=0.5,
                    bgcolor="rgba(255,255,255,0.95)",
                    bordercolor="#E2E8F0",
                    borderwidth=1,
                    font=dict(size=11, family=FONT_FAMILY)
                )
            )
            out_m_ecdf = env_dir / "target_precision_ecdf.png"
            fig_ecdf.write_image(str(out_m_ecdf), scale=3)

print("✅ 6-Panel Convergence and ECDF profiles generated with clean margins and uncollided legends.")


2026-08-26 22:28:03 INFO TemporaryDirectory.cleanup() worked.
2026-08-26 22:28:03 INFO shutil.rmtree worked.
2026-08-26 22:28:03 INFO TemporaryDirectory.cleanup() worked.
2026-08-26 22:28:03 INFO shutil.rmtree worked.
2026-08-26 22:28:03 INFO Chromium init'ed with kwargs {}
2026-08-26 22:28:03 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:28:03 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp59q2w1ox.
2026-08-26 22:28:03 INFO Opening browser.
2026-08-26 22:28:03 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpw5jsbubg.
2026-08-26 22:28:03 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpw5jsbubg
2026-08-26 22:28:04 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp59q2w1ox/index.html
2026-08-26 22:28:05 INFO Getting tab from queue (has 1)
2026-08-26 22:28:05 INFO Got 734B
2026-08-26 22:28:05 INFO Reloading

✅ 6-Panel Convergence and ECDF profiles generated in figures/profiles/


In [20]:
# ── THESIS Figure 6: Cross-Environment Noise Robustness Profile ──────────────
for dim in all_dims:
    valid_solvers, clean_rates, noisy_rates, deltas = service.compute_robustness_profile(
        all_benchmark_data, dim, ALL_SOLVERS_ORDER, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
    )
    fig6 = go.Figure()
    fig6.add_trace(go.Bar(name=f"Deterministic (σ={clean_std})", x=valid_solvers, y=clean_rates, marker=dict(color="#1E3A8A", line=dict(color="#0F172A", width=1.2))))
    fig6.add_trace(go.Bar(name=f"Noisy Stochastic (σ={noisy_std})", x=valid_solvers, y=noisy_rates, marker=dict(color="#EA580C", pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8), line=dict(color="#7C2D12", width=1.2))))
    for s, c_r, n_r, delta in zip(valid_solvers, clean_rates, noisy_rates, deltas):
        drop_pct = (delta / c_r * 100) if c_r > 0 else 0.0
        fig6.add_annotation(x=s, y=max(c_r, n_r) + 0.04, text=f"<b>-Δ{drop_pct:.0f}%</b>" if delta > 0 else "<b>0%</b>", showarrow=False, font=dict(size=10, color="#EA580C" if drop_pct > 25 else "#16A34A", family=FONT_FAMILY))
    fig6.update_layout(
        template="plotly_white",
        title=dict(text=f"<b>Figure 6: Cross-Environment Noise Robustness Profile — {dim}D</b><br><sup>Generalization Retention: Clean vs. Noisy Target Success Rate (Δy ≤ 10⁻⁸) with Performance Drop Badges</sup>", font=dict(size=14, color="#1E293B", family=FONT_FAMILY), x=0.02, y=0.96),
        xaxis=dict(title="<b>Optimization Solver</b>", tickangle=-30, tickfont=dict(size=10, family=FONT_FAMILY)),
        yaxis=dict(title="<b>Overall Target Success Rate</b>", range=[0, 1.15], showgrid=True, gridcolor="#F1F5F9"),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1120, height=600,
        margin=dict(l=75, r=40, t=100, b=120),
        legend=dict(orientation="h", yanchor="top", y=-0.22, xanchor="center", x=0.5, bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1, font=dict(size=11, family=FONT_FAMILY))
    )
    out_p = NOISE_DIR / f"fig_06_robustness_profile_{dim}D.png"
    fig6.write_image(str(out_p), scale=3)

print("✅ Figure 6 (Robustness Profile) generated in results/publication/noise_robustness/")


2026-08-26 22:28:59 INFO TemporaryDirectory.cleanup() worked.
2026-08-26 22:28:59 INFO shutil.rmtree worked.
2026-08-26 22:28:59 INFO Chromium init'ed with kwargs {}
2026-08-26 22:28:59 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:28:59 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpz2wlx_3p.
2026-08-26 22:28:59 INFO Opening browser.
2026-08-26 22:28:59 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmph_4u68cj.
2026-08-26 22:28:59 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmph_4u68cj
2026-08-26 22:29:00 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpz2wlx_3p/index.html
2026-08-26 22:29:00 INFO Getting tab from queue (has 1)
2026-08-26 22:29:00 INFO Got F2A0
2026-08-26 22:29:00 INFO Reloading tab F2A0 before return.
2026-08-26 22:29:00 INFO Putting tab F2A0 back (queue size: 0).
2026-08-26 22:29:00 

✅ Figure 6 (Robustness Profile) generated in results/publication/noise_robustness/


In [21]:
# ── THESIS Figure 3: Prompt Strategy & Model Scale Ablation ──────────────────
for dim in all_dims:
    fig3 = go.Figure()
    for model_name, s_list in MODELS_TO_SOLVERS.items():
        strat_labels, clean_rates, noisy_rates = service.compute_scaffolding_ablation(
            all_benchmark_data, dim, s_list, PROBLEM_IDS, clean_std=clean_std, noisy_std=noisy_std
        )
        is_14b = "14B" in model_name
        color_c = "#0284C7" if is_14b else "#7DD3FC"
        color_n = "#D97706" if is_14b else "#FCD34D"
        fig3.add_trace(go.Bar(name=f"{model_name} (Clean)", x=strat_labels, y=clean_rates, marker=dict(color=color_c, line=dict(color="#0F172A", width=1.0))))
        fig3.add_trace(go.Bar(name=f"{model_name} (Noisy)", x=strat_labels, y=noisy_rates, marker=dict(color=color_n, pattern=dict(shape="/", fillmode="replace", fgcolor="#FFFFFF", fgopacity=0.35, size=8), line=dict(color="#7C2D12", width=1.0))))
    fig3.update_layout(
        template="plotly_white",
        title=dict(
            text=f"<b>Figure 3: Prompt Strategy & Model Scale Ablation — {dim}D</b><br><sup>Empirical Target Success Rate (Δy ≤ 10⁻⁸) by Prompt Scaffolding and Model Scale in Clean vs. Noisy Regimes</sup>",
            font=dict(size=14, color="#1E293B", family=FONT_FAMILY), x=0.02, y=0.96
        ),
        xaxis=dict(title="<b>Prompt Scaffolding Strategy</b>", tickfont=dict(size=11, family=FONT_FAMILY)),
        yaxis=dict(title="<b>Target Success Rate</b>", range=[0, 1.10], showgrid=True, gridcolor="#F1F5F9"),
        barmode="group", bargap=0.25, bargroupgap=0.1,
        width=1100, height=560,
        margin=dict(l=75, r=40, t=100, b=90),
        legend=dict(
            orientation="h", yanchor="top", y=-0.16, xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.95)", bordercolor="#E2E8F0", borderwidth=1,
            font=dict(size=11, family=FONT_FAMILY)
        )
    )
    out_p = ABLATION_DIR / f"fig_03_prompt_strategy_ablation_{dim}D.png"
    fig3.write_image(str(out_p), scale=3)

print("✅ Figure 3 (Prompt Strategy Ablation) generated in results/publication/ablation/")


2026-08-26 22:29:05 INFO Chromium init'ed with kwargs {}
2026-08-26 22:29:05 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:29:05 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpgeit4sxn.
2026-08-26 22:29:05 INFO Opening browser.
2026-08-26 22:29:05 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpwjt64pcw.
2026-08-26 22:29:05 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpwjt64pcw
2026-08-26 22:29:06 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpgeit4sxn/index.html
2026-08-26 22:29:07 INFO Getting tab from queue (has 1)
2026-08-26 22:29:07 INFO Got 72F4
2026-08-26 22:29:07 INFO Reloading tab 72F4 before return.
2026-08-26 22:29:07 INFO Putting tab 72F4 back (queue size: 0).
2026-08-26 22:29:07 INFO Waiting for all cleanups to finish.
2026-08-26 22:29:07 INFO Exiting Kaleido.
2026-08-26 22:29:07 INFO T

✅ Figure 3 (Prompt Strategy Ablation) generated in results/publication/ablation/


In [22]:
# ── THESIS Figure 4: Vargha-Delaney Effect Size (A12) Heatmap ────────────────
a12_matrix = np.full((len(ALL_SOLVERS_ORDER), len(ALL_SOLVERS_ORDER)), 0.5)
for i, s1 in enumerate(ALL_SOLVERS_ORDER):
    for j, s2 in enumerate(ALL_SOLVERS_ORDER):
        if i == j:
            a12_matrix[i, j] = 0.5
        else:
            sub = df_pairwise[(df_pairwise["Solver 1"] == s1) & (df_pairwise["Solver 2"] == s2)]
            if not sub.empty:
                a12_matrix[i, j] = sub["A12"].mean()
            else:
                sub_rev = df_pairwise[(df_pairwise["Solver 1"] == s2) & (df_pairwise["Solver 2"] == s1)]
                if not sub_rev.empty:
                    a12_matrix[i, j] = 1.0 - sub_rev["A12"].mean()

fig4_a12 = go.Figure(data=go.Heatmap(
    z=a12_matrix, x=ALL_SOLVERS_ORDER, y=ALL_SOLVERS_ORDER, colorscale="RdBu_r", zmid=0.5, zmin=0.0, zmax=1.0,
    text=[[f"{val:.2f}" for val in row] for row in a12_matrix], texttemplate="%{text}", textfont=dict(size=10, family=FONT_FAMILY),
    colorbar=dict(title="<b>Â₁₂ Metric</b>", tickvals=[0.0, 0.29, 0.5, 0.71, 1.0], ticktext=["0.0 (Col Large)", "0.29 (Col Med)", "0.50 (Tie)", "0.71 (Row Med)", "1.0 (Row Large)"], thickness=14, len=0.85)
))
fig4_a12.update_layout(
    title=dict(text="<b>Figure 4: Global Vargha-Delaney Effect Size (Â₁₂) Heatmap</b><br><sup>Pairwise Non-Parametric Effect Sizes Averaged Across All 30 Problem Conditions (Row vs. Column)</sup>", font=dict(size=14, color="#1E293B", family=FONT_FAMILY), x=0.02, y=0.96),
    template="plotly_white", width=980, height=820, margin=dict(l=180, r=40, t=95, b=140),
    xaxis=dict(tickangle=-35, tickfont=dict(size=10, family=FONT_FAMILY)), yaxis=dict(tickfont=dict(size=10, family=FONT_FAMILY), autorange="reversed")
)
out_fig4 = EFFECT_SIZES_DIR / "fig_04_a12_heatmap.png"
fig4_a12.write_image(str(out_fig4), scale=3)
print("✅ Figure 4 (A12 Heatmap) generated in results/publication/effect_sizes/")


2026-08-26 22:29:11 INFO Chromium init'ed with kwargs {}
2026-08-26 22:29:11 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:29:11 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpyaq2znz3.
2026-08-26 22:29:11 INFO Opening browser.
2026-08-26 22:29:11 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp0o6ox8of.
2026-08-26 22:29:11 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp0o6ox8of
2026-08-26 22:29:12 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpyaq2znz3/index.html
2026-08-26 22:29:13 INFO Getting tab from queue (has 1)
2026-08-26 22:29:13 INFO Got 35DC
2026-08-26 22:29:13 INFO Reloading tab 35DC before return.
2026-08-26 22:29:13 INFO Putting tab 35DC back (queue size: 0).
2026-08-26 22:29:13 INFO Waiting for all cleanups to finish.
2026-08-26 22:29:13 INFO Exiting Kaleido.
2026-08-26 22:29:13 INFO T

✅ Figure 4 (A12 Heatmap) generated in results/publication/effect_sizes/


In [23]:
# ── THESIS Figure 7: Pairwise Win / Tie / Loss Summary Ranking ────────────────
win_counts = df_pairwise[df_pairwise["Significant (FDR)"] == True]["Outcome"].value_counts()
win_data = []
for s in ALL_SOLVERS_ORDER:
    w_cnt = win_counts.get(f"{s} Wins", 0)
    win_data.append({"Solver": s, "FDR Wins": w_cnt, "Type": "Classical" if " / " not in s else "LLaMEA Evolved"})
df_wins = pd.DataFrame(win_data).sort_values(by="FDR Wins", ascending=True)

fig7 = go.Figure(go.Bar(
    x=df_wins["FDR Wins"], y=df_wins["Solver"], orientation="h",
    marker=dict(color=[get_solver_color(s) for s in df_wins["Solver"]], line=dict(color="#0F172A", width=1.0)),
    text=df_wins["FDR Wins"], textposition="outside", textfont=dict(size=11, family=FONT_FAMILY, color="#1E293B")
))
fig7.update_layout(
    template="plotly_white",
    title=dict(text="<b>Figure 7: Global Pairwise Win Summary (Mann-Whitney U with FDR Correction)</b><br><sup>Total Significant Head-to-Head Victories (p < 0.05) Across All 30 BBOB Problem Conditions (1,630 Total Comparisons)</sup>", font=dict(size=14, color="#1E293B", family=FONT_FAMILY), x=0.02, y=0.96),
    xaxis=dict(title="<b>Statistically Significant Wins (FDR Adjusted)</b>", range=[0, 210], showgrid=True, gridcolor="#F1F5F9"),
    yaxis=dict(tickfont=dict(size=11, family=FONT_FAMILY)),
    width=950, height=540, margin=dict(l=180, r=40, t=95, b=65)
)
out_fig7 = MAIN_RESULTS_DIR / "fig_07_win_tie_loss.png"
fig7.write_image(str(out_fig7), scale=3)
print("✅ Figure 7 (Win/Tie/Loss Ranking) generated in results/publication/main_results/")


2026-08-26 22:29:13 INFO Chromium init'ed with kwargs {}
2026-08-26 22:29:13 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:29:13 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp1u1d3ke8.
2026-08-26 22:29:13 INFO Opening browser.
2026-08-26 22:29:13 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpov55xec0.
2026-08-26 22:29:13 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpov55xec0
2026-08-26 22:29:14 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp1u1d3ke8/index.html
2026-08-26 22:29:15 INFO Getting tab from queue (has 1)
2026-08-26 22:29:15 INFO Got 37D4
2026-08-26 22:29:15 INFO Reloading tab 37D4 before return.
2026-08-26 22:29:15 INFO Putting tab 37D4 back (queue size: 0).
2026-08-26 22:29:15 INFO Waiting for all cleanups to finish.
2026-08-26 22:29:15 INFO Exiting Kaleido.
2026-08-26 22:29:15 INFO T

✅ Figure 7 (Win/Tie/Loss Ranking) generated in results/publication/main_results/


In [24]:
# ── THESIS Figure 8: Synthesis vs. Evaluation Transferability ────────────────
r_val, p_val = service.compute_synthesis_transfer_correlation(df_exp)
fig8 = go.Figure()
fig8.add_trace(go.Scatter(
    x=df_exp["best_final_error"], y=df_exp["best_final_error"],
    mode="markers", marker=dict(size=8, color="#0284C7", opacity=0.7, line=dict(color="#0F172A", width=0.8)),
    name="Evolutionary Runs"
))
fig8.update_layout(
    template="plotly_white",
    title=dict(text=f"<b>Figure 8: Synthesis Horizon vs. Benchmark Transferability</b><br><sup>Pearson Correlation r = {r_val:.3f} (p = {p_val:.3e}) Across 292 Synthesis Runs</sup>", font=dict(size=14, color="#1E293B", family=FONT_FAMILY), x=0.02, y=0.96),
    xaxis=dict(type="log", title="<b>Synthesis Error (1,000 Budget)</b>", showgrid=True, gridcolor="#F1F5F9"),
    yaxis=dict(type="log", title="<b>Benchmark Error (50,000 Budget)</b>", showgrid=True, gridcolor="#F1F5F9"),
    width=850, height=500, margin=dict(l=65, r=30, t=95, b=65)
)
out_fig8 = MAIN_RESULTS_DIR / "fig_08_synthesis_transfer.png"
fig8.write_image(str(out_fig8), scale=3)
print("✅ Figure 8 (Synthesis Transferability) generated in results/publication/main_results/")


2026-08-26 22:29:15 INFO Chromium init'ed with kwargs {}
2026-08-26 22:29:15 INFO Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-08-26 22:29:15 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpyxu6b4km.
2026-08-26 22:29:15 INFO Opening browser.
2026-08-26 22:29:15 INFO Temp directory created: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp8que15yk.
2026-08-26 22:29:15 INFO Temporary directory at: /var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmp8que15yk
2026-08-26 22:29:16 INFO Conforming 1 to file:///var/folders/sl/f2m6tw2d2d79p7ly8v3dfmd00000gn/T/tmpyxu6b4km/index.html
2026-08-26 22:29:17 INFO Getting tab from queue (has 1)
2026-08-26 22:29:17 INFO Got 8E57
2026-08-26 22:29:17 INFO Reloading tab 8E57 before return.
2026-08-26 22:29:17 INFO Putting tab 8E57 back (queue size: 0).
2026-08-26 22:29:17 INFO Waiting for all cleanups to finish.
2026-08-26 22:29:17 INFO Exiting Kaleido.
2026-08-26 22:29:17 INFO T

✅ Figure 8 (Synthesis Transferability) generated in results/publication/main_results/
